#### Problem statement:
Our goal is to develop a machine learning-based movie recommendation system that can predict and recommend movies to users based on their unique preferences. This system would use Term Frequency-Inverse Document Frequency (TF-IDF) vectorization technique to transform text data into meaningful numerical vectors, and cosine similarity to compute the similarity between these vectors.

#### Importing the necessary libraries:

In [19]:
import pandas as pd
import numpy as np
import ast
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

#### Loading the dataset:

In [20]:
movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")
df = movies.merge(credits, on="title")

In [21]:
df.sample(5)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
4702,8000,"[{""id"": 18, ""name"": ""Drama""}, {""id"": 10749, ""n...",http://weekenderfilm.tumblr.com/,79120,"[{""id"": 237, ""name"": ""gay""}, {""id"": 1025, ""nam...",en,Weekend,After a drunken house party with his straight ...,1.041254,"[{""name"": ""EM Media"", ""id"": 1917}, {""name"": ""T...",...,96.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,A (sort of) love story between two guys over a...,Weekend,7.4,163,79120,"[{""cast_id"": 1000, ""character"": ""Russell"", ""cr...","[{""credit_id"": ""52fe49c2c3a368484e13e4e1"", ""de..."
4792,0,"[{""id"": 35, ""name"": ""Comedy""}, {""id"": 10749, ""...",NaN,44990,"[{""id"": 10183, ""name"": ""independent film""}]",en,Breaking Upwards,"'Breaking Upwards' explores a young, real-life...",0.674570,[],...,88.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,NaN,Breaking Upwards,5.6,12,44990,"[{""cast_id"": 1, ""character"": ""Zoe"", ""credit_id...","[{""credit_id"": ""572239efc3a368712a0007db"", ""de..."
2976,12000000,"[{""id"": 36, ""name"": ""History""}, {""id"": 18, ""na...",http://www.cristiadafilm.com/,96399,"[{""id"": 179431, ""name"": ""duringcreditsstinger""}]",en,For Greater Glory - The True Story of Cristiada,"A chronicle of the Cristeros War (1926-1929), ...",5.759545,"[{""name"": ""Dos Corazones"", ""id"": 41537}]",...,145.0,"[{""iso_639_1"": ""es"", ""name"": ""Espa\u00f1ol""}, ...",Released,The True Story of Cristiada,For Greater Glory - The True Story of Cristiada,6.4,35,96399,"[{""cast_id"": 2, ""character"": ""Tulita"", ""credit...","[{""credit_id"": ""52fe49b29251416c750d0649"", ""de..."
629,66000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",NaN,136797,"[{""id"": 9666, ""name"": ""street race""}, {""id"": 1...",en,Need for Speed,The film revolves around a local street-racer ...,54.814890,"[{""name"": ""DreamWorks SKG"", ""id"": 27}, {""name""...",...,130.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,For honor. For love. For redemption.,Need for Speed,6.1,1520,136797,"[{""cast_id"": 8, ""character"": ""Tobey Marshall"",...","[{""credit_id"": ""53c1c8f50e0a26158f0098bd"", ""de..."
4139,0,"[{""id"": 27, ""name"": ""Horror""}, {""id"": 35, ""nam...",http://www.lesbianvampirekillersmovie.com/,18238,"[{""id"": 293, ""name"": ""female nudity""}, {""id"": ...",en,Lesbian Vampire Killers,With their women having been enslaved by a pac...,6.632891,"[{""name"": ""AV Pictures"", ""id"": 3471}]",...,86.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"After Twilight, The Real Party Starts.",Lesbian Vampire Killers,5.3,124,18238,"[{""cast_id"": 1, ""character"": ""Fletch"", ""credit...","[{""credit_id"": ""52fe476c9251416c75098cdf"", ""de..."


#### Data cleaning and preprocessing:

In [22]:
#Information about the dataset:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

In [23]:
#Removing the unnecessary columns:
df = df[["genres", "id", "keywords", "title", "overview", "cast", "crew"]]

In [24]:
#Checking for and handling missing entries:
df.isnull().sum()
df.dropna(inplace=True)

In [25]:
#Checking for and handling duplicates:
df.duplicated().sum()

np.int64(0)

In [26]:
#Function to extract the list of genres and list of keywords:
def genre_keyword_converter(obj):
    L = []
    for d in ast.literal_eval(obj):
        L.append(d["name"])
    return L

df["genres"] = df["genres"].apply(genre_keyword_converter)
df["keywords"] = df["keywords"].apply(genre_keyword_converter)

In [27]:
#Function to extract the list of cast:
def cast_converter(obj):
    L = []
    c=0
    for d in ast.literal_eval(obj):
        if c==3:
            break
        L.append(d["name"])
        c += 1
    return L

df["cast"] = df["cast"].apply(cast_converter)

In [28]:
#Function to extract the name of the director:
def crew_converter(obj):
    for d in ast.literal_eval(obj):
        if d["job"] == "Director":
            return [d["name"]]
        
df["director"] = df["crew"].apply(crew_converter)
df = df.drop(columns="crew")

In [29]:
#Converting the "overview" column to a list:
df["overview"] = df["overview"].apply(lambda x: x.split(" "))

In [30]:
#Eliminating spaces between keywords/tags:
def replace(L):
    if not L:
        return []
    else:
        return [i.replace(" ", "") for i in L]
df["genres"] = df["genres"].apply(replace)
df["keywords"] = df["keywords"].apply(replace)
df["director"] = df["director"].apply(replace)
df["cast"] = df["cast"].apply(replace)

In [31]:
df.sample(10)

,genres,id,keywords,title,overview,cast,director
2935,"[Drama, Horror, Thriller, Crime]",13937,[],Raising Cain,"[When, neighborhood, kids, begin, vanishing,, ...","[JohnLithgow, LolitaDavidovich, StevenBauer]",[BrianDePalma]
2407,"[Horror, Thriller, Mystery]",3597,"[secret, blackmail, fisherman, police, highsch...",I Know What You Did Last Summer,"[As, they, celebrate, their, high, school, gra...","[JenniferLoveHewitt, SarahMichelleGellar, Ryan...",[JimGillespie]
1686,"[Drama, Music, Romance]",69,"[germany, prison, musicrecord, adultery, count...",Walk the Line,"[A, chronicle, of, country, music, legend, Joh...","[JoaquinPhoenix, ReeseWitherspoon, GinniferGoo...",[JamesMangold]
2570,"[Drama, Fantasy, Horror, Thriller]",9100,"[witch, suicideattempt, puberty, magic, blackm...",The Craft,"[A, Catholic, school, newcomer, falls, in, wit...","[RobinTunney, FairuzaBalk, NeveCampbell]",[AndrewFleming]
520,[Comedy],38365,"[overweight, swing, foot, convertible, arrow]",Grown Ups,"[After, their, high, school, basketball, coach...","[AdamSandler, SalmaHayek, MariaBello]",[DennisDugan]
4713,"[Thriller, Drama, Horror]",50497,"[newspaper, exorcism, poltergeist, priest, hau...",When the Lights Went Out,"[Yorkshire,, 1974,, the, Maynard, family, move...","[KateAshfield, JoHartley, MartinCompston]",[PatHolden]
602,"[Action, Adventure, Thriller]",10550,"[lossoffamily, enemy, adversary, agent]",Ballistic: Ecks vs. Sever,"[Jonathan, Ecks,, an, FBI, agent,, realizes, t...","[AntonioBanderas, LucyLiu, GreggHenry]",[WychKaosayananda]
1964,"[Action, Drama, Western]",174751,[],Jane Got a Gun,"[After, her, outlaw, husband, returns, home, s...","[NataliePortman, JoelEdgerton, EwanMcGregor]",[GavinO'Connor]
27,"[Thriller, Action, Adventure, ScienceFiction]",44833,"[fight, u.s.navy, mindreading, hongkong, socce...",Battleship,"[When, mankind, beams, a, radio, signal, into,...","[TaylorKitsch, AlexanderSkarsgård, Rihanna]",[PeterBerg]
1315,"[Drama, Thriller]",11306,"[newyork, surgeon, british, suspense, morgue, ...",Extreme Measures,"[Thriller, about, Guy, Luthan, (Hugh, Grant),,...","[HughGrant, GeneHackman, SarahJessicaParker]",[MichaelApted]


In [32]:
#Concatenating keywords into the "tags" column and dropping unnecessary columns:
df["tags"] = df["keywords"] + df["overview"] + df["cast"] + df["director"] + df["genres"]
df.drop(columns=["keywords", "genres", "overview", "director", "cast"], inplace=True)

In [33]:
df["tags"] = df["tags"].apply(lambda x: " ".join(x))
df ["tags"] = df["tags"].apply(lambda x: x.lower())

In [34]:
#Function to perform stemming and remove stopwords from a piece of text:
# stemmer = PorterStemmer()
# stop_words = set(stopwords.words('english'))
# def preprocess_text(text):
#     words = word_tokenize(text)
#     processed_words = [stemmer.stem(word) for word in words if word not in stop_words]
#     return ' '.join(processed_words)

# df["tags"] = df["tags"].apply(preprocess_text)


import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import TreebankWordTokenizer

# Download only the required resources
nltk.download('stopwords')

# Initialize stemmer and tokenizer
stemmer = PorterStemmer()
stop_words = set(stopwords.words('english'))
tokenizer = TreebankWordTokenizer()

# Define preprocessing function
def preprocess_text(text):
    words = tokenizer.tokenize(text)
    processed_words = [stemmer.stem(word) for word in words if word.lower() not in stop_words]
    return ' '.join(processed_words)

# Apply preprocessing
df["tags"] = df["tags"].apply(preprocess_text)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\annie\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [35]:
df.sample(10)

,id,title,tags
341,32657,Percy Jackson & the Olympians: The Lightning T...,monster greekmytholog god poseidon lightningbo...
146,80321,Madagascar 3: Europe's Most Wanted,"madagascar 3d alex , marti , gloria melman sti..."
4333,53502,The Dead Undead,vampir zombi vanityproject madcowdiseas good v...
3797,111190,Adore,lover womandirector lil roz two lifelong frien...
526,1125,Dreamgirls,musicrecord manag blackpeopl adulteri soul sho...
2177,526,Ladyhawke,moon monk swordplay bishop cathedr falcon twil...
913,10201,Yes Man,bungee-jump scooter carl allen stumbl across w...
598,80585,Rock of Ages,music rocker teenag younglov rocksta small tow...
2269,241257,Regression,"investig memoryloss minnesota , 1990. detect b..."
3668,340816,Christmas Eve,photograph surgeon orchestra doctor caraccid p...


In [36]:
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(df["tags"])
cs = cosine_similarity(tfidf_matrix, tfidf_matrix)

#### Exporting the cosine similarity table and the processed dataframe:

In [37]:
pickle.dump(df.to_dict(), open("movies.pkl", "wb"))
pickle.dump(cs, open("cs.pkl", "wb"))